# Using ETSI MEC profile of CAPIF  in CAPIF application

## Introduction

3GPP CAPIF (Common API Framework) is a standardized API management framework designed to enable a unified northbound API approach across 3GPP network functions (see 3GPP TS 23.222 version 18.6.0 Release 18/ETSI TS 123 222 V18.6.0 (2024-06) and 3GPP TS 29.222 version 18.6.0 Release 18/ETSI TS 129 222 V18.6.0 (2022-06)).

This tutorial introduces the step by step procedure to create a basic CAPIF application to exploit the ETSI MEC CAPIF profile as described in ETSI GS MEC 011 (V3.2.1) Clause 9.
It uses the ETSI MEC Sandbox simulator.

<div class="alert alert-block alert-danger">
    <b>Note:</b> These source code examples are simplified and ignore return codes and error checks to a large extent. We do this to highlight how to use the MEC Sandbox API and the different MEC satndards and reduce unrelated code.
A real-world application will of course properly check every return value and exit correctly at the first serious error.
</div>


## The basics of developing a MEC application


<div class="alert alert-warning" role="alert">
    <b>Note:</b> The sub-paragraph 'Putting everything together' is a specific paragraph where all the newly features introduced in the main paragraph are put together to create an executable block of code. It is possible to skip this block of code by removing the comment character (#) on first line of this block of code.
</div>

Before going to create our CAPIF application skeleton, the following steps shall be done:

1) Apply the python imports

In [ ]:
from __future__ import division # Import floating-point division (1/4=0.25) instead of Euclidian division (1/4=0)

import os
import sys
import re
import logging
import threading
import time
import json
import uuid
import base64

import pprint

import requests

from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, HTTPServer

try:
    import urllib3
except ImportError:
    raise ImportError('Swagger python client requires urllib3.')


The following imports are required to support the security aspects such as certificates management, signatures...

In [ ]:
from OpenSSL.SSL import FILETYPE_PEM
from OpenSSL.crypto import (dump_certificate_request, dump_privatekey, load_publickey, PKey, TYPE_RSA, X509Req, dump_publickey)


2) Initialize of the global constants (cell 3)

In [ ]:
REGISTER_HOSTNAME     = 'lab-oai.etsi.org'                    # capif-prev.mobilesandbox.cloud
REGISTER_PORT         = 31120                                 # 36212
REGISTER_USER         = 'admin'                               # Basic AUTH for registration
REGISTER_PASSWORD     = 'password123'                         # Basic AUTH for registration

CAPIF_HOSTNAME        = 'lab-oai.etsi.org'
CAPIF_PORT            = 443

USER_PASSWORD         = 'password123'

TRY_MEC_URL           = 'try-mec.etsi.org'                     # MEC Sandbox URL
TRY_MEC_SESSION_ID    = 'sbxgs9x587'                           # MEC Sandbox identifier
TRY_MEC_PLTF          = 'mep1'                                 # MEC Platform identifier (depending of the network scenario loaded)
MEC_APP_INST_ID       = 'f1e4d448-e277-496b-bf63-98391cfd20fb' # A MEC application identifier


3) Setup the logger instance and the HTTP REST API (cell 4)

In [ ]:
# Initialize the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logging.basicConfig(filename='/tmp/' + time.strftime('%Y%m%d-%H%M%S') + '.log')
l = logging.StreamHandler()
l.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
logger.addHandler(l)


4) Setup the global variables (cell 5)

In [ ]:
# Initialize the global variables
ca_root                = "" # The CAPIF root certificate
ccf_api_onboarding_url = "" #
ccf_publish_url        = "" # The CAPIF publish API endpoint
ccf_discover_url       = "" # The CAPIF discovery endpoint
ccf_security_url       = "" # The CAPIF security endpoint
ccf_onboarding_url     = "" # The CAPIF onboarding endpoint


To enable the Automatic Debugger Calling, uncomment the code bellow.

In [ ]:
#!pip install ipdb
#import ipdb
#%pdb on
# Use the command ipdb.set_trace() to set a breakpoint

## Create our first CAPIF application

The first step to develop a MEC application is to create the application skeleton which contains the minimum steps below:

- Login to instanciate a MEC Sandbox
- Logout to delete a existing MEC Sandbox

#### Login

The login operation is required by ETSI TS 123 222 V18.6.0 (2024-06) Clause 4.5 Operations, Administration and Maintenance but is out of the scope of ETSI TS 129 222 V18.6.0 (2022-06).

In [ ]:
def process_login() -> tuple:
    """
    Logs in to the CAPIF server.
    :return A dictionary containing the login response, or None if login fails
    """
    global logger

    logger.debug('>>> process_login')

    try:
        url = 'https://' + REGISTER_HOSTNAME + ':' + str(REGISTER_PORT) + '/login'
        logger.debug('process_login: url=' + url)
        auth_string = f"{REGISTER_USER}:{REGISTER_PASSWORD}"
        encoded_auth = base64.b64encode(auth_string.encode('utf-8')).decode('utf-8')
        headers = {'Content-Type': 'application/json', 'Authorization': f'Basic {encoded_auth}'}
        logger.debug('process_login (step1): headers: ' + str(headers))
        response = requests.post(url, headers=headers, verify=False)
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('process_login (step2): result: ' + str(response.json()))
        if response.status_code != 200:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return None, None
        tokens = json.loads(response.text)
        return tokens['refresh_token'], tokens['access_token']
    except requests.exceptions.RequestException as e:
        logger.error(f"Login failed: {e}")

    return None, None
    # End of function process_login

### Logout

The logout operation is required by ETSI TS 123 222 V18.6.0 (2024-06) Clause 4.5 Operations, Administration and Maintenance but is out of the scope of ETSI TS 129 222 V18.6.0 (2022-06).

In [ ]:
def process_logout():
    """
    Logs out from the CAPIF server
    Nothing to do
    """
    pass
    # End of function process_logout

### Putting everything together
Now, it is time now to create the our first iteration of our CAPIF/MEC application. Here the logic is:
*   Login
*   Print obtained tokens
*   Logout
*   Check that logout is effective

In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the first sprint of our CAPIF application:
        - Login
        - Print obtained tokens
        - Logout
        - Check that logout is effective
    """
    global logger

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Login
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        return

    # Print obtained tokens
    logger.debug("Login successful: admin_token=" + admin_token)

    # Logout
    process_logout()

    # Check that logout is effective
    logger.debug('To check that logout is effective')

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Create the API Provider

The next step is to create a new user associated to our CAPIF application to obtain a user UUID. It will be used to genereate ceertificates to be used during TLS mutual authentication and API onboarding and offboarding for instance.

**Note:** It is required by ETSI TS 123 222 V18.6.0 (2024-06) Clause 4.5 Operations, Administration and Maintenance but is out of the scope of ETSI TS 129 222 V18.6.0 (2022-06).


### Creating a new user

The cell below provides an implementation for this user creation.

**Note:** To improve this code, the user profile shlould be fully parametrized.

In [ ]:
def create_user(p_admin_token: str) -> tuple:
    """
    Creates a new user.
    :return: The user UUID on success, None otherwise
    """
    global logger

    logger.debug('>>> create_user')

    try:
        user_name = str(uuid.uuid1())
        url = 'https://' + REGISTER_HOSTNAME + ':' + str(REGISTER_PORT) + '/createUser'
        logger.debug('create_user: url=' + url)
        headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + p_admin_token}
        logger.debug('create_user (step1): headers: ' + str(headers))
        data = {
            'username': user_name,
            'password': USER_PASSWORD,
            'enterprise': 'ETSI',
            'country': 'France',
            'email': 'ocf@etsi.org',
            'purpose': 'Tutorial on MEC/OpenCAPIF',
            'phone_number': "+330405060708",
            'company_web': 'www.etsi.org',
            'description': 'A step by step procedure to create a basic CAPIF application to exploit the ETSI MEC CAPIF profile'
        }
        response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)
        logger.debug('create_user (step2): response=' + str(response))
        response.raise_for_status()  # Raise an exception for bad status codes
        if response.status_code != 201:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return ()
        tokens = json.loads(response.text)
        return (user_name, tokens['uuid'])
    except requests.exceptions.RequestException as e:
        logger.error(f"Error creating user: {e}")

    return ()
    # End of function create_user

### Deleting an existing User

Before to terminate our CAPIF application, we have to clean up the resources. So, a function to delete a created user is required.

In [ ]:
def delete_user(p_user_uuid: str, p_admin_token: str) -> int:
    """
    Deletes a user.
    :param p_user_uuid: The user UUID
    :return: 0 on success, -1 otherwise
    """
    global logger

    logger.debug('>>> delete_user')

    try:
        url = 'https://' + REGISTER_HOSTNAME + ':' + str(REGISTER_PORT) + '/deleteUser/' + p_user_uuid
        logger.debug('delete_user: url=' + url)
        headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + p_admin_token}
        response = requests.delete(url, headers=headers, verify=False)
        logger.debug('delete_user: response=' + str(response))
        response.raise_for_status()  # Raise an exception for bad status codes
        return 0
    except requests.exceptions.RequestException as e:
        logger.error(f"Error creating user: {e}")

    return -1
    # End of function delete_user

### Putting everything together
It is time now to create the our second iteration of our CAPIF/MEC application.

The sequence is the following:
*   Login
*   Print obtained tokens
*   Create a new user
*   Print the user UUID
*   Delete the newly created user
*   Logout


In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the second sprint of our CAPIF/MEC application:
        - Login
        - Print obtained tokens
        - Create a new user
        - Print the user UUID
        - Delete the newly created user
        - Logout
    """
    global logger

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Login
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        return

    # Print obtained tokens
    logger.debug("Login successful: admin_token=" + admin_token)

    # Create a new user
    user_name, user_uuid = create_user(admin_token)
    if len(user_uuid) == 0:
        return

    # Print User UUID
    logger.debug("User successfully created: user_uuid=" + user_uuid)

    time.sleep(5) # Sleep for 5 seconds

    # Delete the newly created user
    delete_user(user_uuid, admin_token)

    # Logout
    process_logout()

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Getting security materials

The purpose is to retrieves peer certificates for the TLS mutual authentication purpose and the JWT token to onboarding and offboarding APIs.
The following information is retrived:
- The root certificate
- An access token which will be used for onboarding and offboarding APIs
- The URLs for the different CAPIF endpoints:
 * API onbording endpoint
 * API discovery endpoint
 * API publish endpoint
 * Security endpoint

This operation needs the user name and the user password used in previous [chapter](#create_the_invoker_/_provider).


### Getting certificates

In [ ]:
def get_auth(p_user_name: str, p_user_password: str) -> dict:
    """
    Gets the authentication information.
    :param The user name
    :param The user password
    :return A dictionary containing the authentication information on success, or an empty dictionary otherwise
    """
    global logger

    logger.debug('>>> get_auth')

    try:
        url = 'https://' + REGISTER_HOSTNAME + ':' + str(REGISTER_PORT) + '/getauth'
        logger.debug('get_auth: url=' + url)
        auth_string = f"{p_user_name}:{p_user_password}"
        encoded_auth = base64.b64encode(auth_string.encode('utf-8')).decode('utf-8')
        headers = {'Content-Type': 'application/json', 'Authorization': f'Basic {encoded_auth}'}
        logger.debug('get_auth (step1): headers: ' + str(headers))
        response = requests.get(url, headers=headers, verify=False)
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('get_auth (step2): result: ' + str(response.json()))
        if response.status_code != 200:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return dict()
        auth = json.loads(response.text)
        return auth
    except requests.exceptions.RequestException as e:
        logger.error(f"get_auth failed: {e}")

    return dict()
    # End of function get_auth

### Putting everything together

Now, it is time now to create the our third iteration of our MEC application. Here the logic is:

- Login
- Create the user
- Get the information to use CAPIF (security materials & URLs)
- Print the information to use CAPI
- Delete the user
- Logout


In [ ]:
%%script echo skipping
# Uncomment the ;line above to skip execution of this cell
def process_main():
    """
    This is the first sprint of our skeleton of our CAPIF application:
        - Login
        - Print obtained tokens
        - Create a new user
        - Get the information to use CAPIF (security materials & URLs)
        - Print the information to use CAPI
        - Delete the newly created user
        - Logout
    """
    global logger, ca_root, ccf_api_onboarding_url, ccf_publish_url, ccf_discover_url, ccf_security_url, ccf_onboarding_url

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Login
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        return

    # Create a new user
    user_name, user_uuid = create_user(admin_token)
    if len(user_uuid) == 0:
        return

    auth = get_auth(user_name, USER_PASSWORD)
    if len(auth) == 0:
        return

    # Print the authentication information
    logger.debug("Authentication information=" + str(auth))
    access_token = auth['access_token']
    ca_root = auth['ca_root']
    ccf_api_onboarding_url = auth['ccf_api_onboarding_url']
    ccf_discover_url = auth['ccf_discover_url']
    ccf_onboarding_url = auth['ccf_onboarding_url']
    ccf_publish_url = auth['ccf_publish_url']
    ccf_security_url = auth['ccf_security_url']

    time.sleep(5) # Sleep for 5 seconds

    # Delete the newly created user
    delete_user(user_uuid, admin_token)

    # Logout
    process_logout()

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Onboarding and offboarding APIs


### Generate certificates

Until now, all HTTPS exchanges were done with the the 'verify' attribute of the HTTP reques set to False. It means that the TLS mutual authentication was disabled.

Fo the process of onboarding and offboarding APIs, the TLS mutual authentication is required. We already got the peer certificate to verify peer but we need to generate our own certificate to be verified by the CAPIF server. This is the purpose of the following functions to generate the public/private keys and generate a CSR and request certificates for each of the three functions AMF, AEF and APF.

**Refer to:** ETSI TS 129 222 V18.6.0 (2022-06) Clauses 5.11 CAPIF_API_Provider_Management and 8.9 CAPIF_API_Provider_Management_API


In [ ]:
def generate_csr(p_cn: str, p_org: str, p_country: str) -> tuple:
    """
    To generate the CSR and generate the dumps
    :param p_cn: The common name
    :param p_org: The organization
    :param p_country: The country
    :return: The CSR and the private keys on success, None otherwise
    """
    global logger

    logger.debug('>>> generate_csr')

    # Generate the public/private key
    key = PKey()
    key.generate_key(TYPE_RSA, 2048)

    # Generate the CSR
    req = X509Req()
    req.get_subject().CN = p_cn
    req.get_subject().O = p_org
    req.get_subject().C = p_country
    req.set_pubkey(key)
    req.sign(key, 'sha256')

    # Generate the dumps
    csr_request = dump_certificate_request(FILETYPE_PEM, req)
    private_key = dump_privatekey(FILETYPE_PEM, key)
    logger.debug('generate_csr: PrivKey: ' + str(private_key))

    return (csr_request, private_key)


**Note:** The function above can be improved using parameter for the SHA, and the signature/encryption algorithm.

### Onboard the API provider

The purpose here is to get certificates from CAPIF in order to export our APIs for the different functions:
- AMF: API Management Function
- AEF: API Exposing Function
- APF: API Publishing Function

In [ ]:
def onboard_provider(p_name: str, p_access_token: str) -> dict:
    """
    To onboard the provider.
    :param p_name: The name of the provider
    :return: A dictionary containing security material for each CAPIF endpoint on success, or an empty dictionary otherwise
    """
    global logger, ccf_api_onboarding_url, access_token

    logger.debug('>>> onboard_provider')

    try:
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_api_onboarding_url
        logger.debug('onboard_provider: url=' + url)
        headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + p_access_token}
        logger.debug('onboard_provider (step1): headers: ' + str(headers))
        # Build the list of certificate request for the three endpoints
        l = []
        amf_csr_request, amf_private_key = generate_csr("AMF", "ETSI", "Fr")
        amf_entry = {
            'regInfo': {
                'apiProvPubKey': amf_csr_request.decode("utf-8")
            },
            'apiProvFuncRole': 'AMF'
        }
        l.append(amf_entry)
        aef_csr_request, aef_private_key = generate_csr("AEF", "ETSI", "Fr")
        aef_entry = {
            'regInfo': {
                'apiProvPubKey': aef_csr_request.decode("utf-8")
            },
            'apiProvFuncRole': 'AEF'
        }
        l.append(aef_entry)
        apf_csr_request, apf_private_key = generate_csr("APF", "ETSI", "Fr")
        apf_entry = {
            'regInfo': {
                'apiProvPubKey': apf_csr_request.decode("utf-8")
            },
            'apiProvFuncRole': 'APF'
        }
        l.append(apf_entry)
        # Build the request body
        data = {
            'apiProvFuncs': l,
            'apiProvDomInfo': p_name,
            'suppFeat': 'fff',
            'failReason': 'string',
            'regSec': p_access_token
        }
        logger.debug('onboard_provider (step2): body: ' + str(data))
        response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('onboard_provider (step3): result: ' + str(response.json()))
        if response.status_code != 201:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return dict()
        res = json.loads(response.text)
        # Add an entry for CSRs and private keys for future usage
        res['csr'] = {
            'amf': [amf_csr_request, amf_private_key],
            'aef': [aef_csr_request, aef_private_key],
            'apf': [apf_csr_request, apf_private_key]
        }
        return res
    except requests.exceptions.RequestException as e:
        logger.error(f"onboard_provider failed: {e}")

    return dict()
    # End of function onboard_provider

### Offboard the API provider

The purpose is to offboard the API provider from the CAPIF server. Here, the certificate and the private key of the AMF endpoint are required (TLS mutual authentication).

In [ ]:
def offboard_provider(p_api_provider_id: str, p_bundle: tuple) -> list:
    """
    To offboard the API provider.
    :param p_api_provider_id: The identifier of the API provider
    :param p_bundle: The bundle of certificates and keys for the TLS mutual authentication operations
    :return: A list containing the files created for the TLS mutual authentication operations on success, or an empty list otherwise
    """
    global logger, ccf_api_onboarding_url, ca_root

    logger.debug('>>> offboard_provider')

    try:
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_api_onboarding_url + '/' + p_api_provider_id
        logger.debug('offboard_provider: url=' + url)
        headers = {'Content-Type': 'application/json'}
        logger.debug('offboard_provider (step1): headers: ' + str(headers))
        bundle = store_certificate_2_files(p_bundle[0], p_bundle[1], ca_root) # Use CA certificate for verif
        if len(bundle) != 3:
            logger.error(f"Error converting in-memory bundle into files")
            return []
        logger.debug('offboard_provider (step2): bundle: ' + str(bundle))
        response = requests.delete(url, headers=headers, cert=(bundle[0], bundle[1]), verify=bundle[2])
        logger.debug('offboard_provider (step3): response=' + str(response))
        response.raise_for_status()  # Raise an exception for bad status codes
        if response.status_code != 204:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return []
    except requests.exceptions.RequestException as e:
        logger.error(f"offboard_provider failed: {e}")
        return []

    return bundle
    # End of function offboard_provider

def store_certificate_2_files(p_certificate, p_private_key, p_ca_root) -> list:
    """
    Save certificate and key into files
    :param p_certificate:
    :param p_private_key:
    :param p_ca_root:
    :return: A list of file paths on success, an empty list otherwise
    """
    global logger

    logger.debug('>>> store_certificate_2_files')
    try:
        with open("p_crt.crt", "w") as f:
            f.write(p_certificate)
        with open("p_key.key", "w") as f:
            f.write(p_private_key.decode('utf-8'))
        with open("ca_root.pem", "w") as f:
            f.write(p_ca_root)
        return ["p_crt.crt", "p_key.key", "ca_root.pem"]
    except Exception as e:
        logger.error(f"An error occurred: {e}")

    return []
    # End of function store_certificate_2_files


### Putting everything together

Now, it is time now to create the our third iteration of our CAPIF/MEC application. Here the logic is:

- Login
- Create the user
- Get the information to use CAPIF (security materials & URLs)
- Onboard the provider
- Print certificates for each function
- Delete the user
- Logout


In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the third sprint of our CAPIF/MEC application:
        - Login
        - Print obtained tokens
        - Create a new user
        - Get the information to use CAPIF (security materials & URLs)
        - Onboard the provider
        - Print certificates for each function
        - Delete the newly created user
        - Logout
    """
    global logger, ca_root, ccf_api_onboarding_url, ccf_publish_url, ccf_discover_url, ccf_security_url, ccf_onboarding_url

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Login
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        return

    # Create a new user
    user_name, user_uuid = create_user(admin_token)
    if len(user_uuid) == 0:
        return

    auth = get_auth(user_name, USER_PASSWORD)
    if len(auth) == 0:
        return

    # Set the CAPIF access information
    access_token = auth['access_token']
    ca_root = auth['ca_root']
    ccf_api_onboarding_url = auth['ccf_api_onboarding_url']
    ccf_discover_url = auth['ccf_discover_url']
    ccf_onboarding_url = auth['ccf_onboarding_url']
    ccf_publish_url = auth['ccf_publish_url']
    ccf_security_url = auth['ccf_security_url']
    logger.debug("ccf_api_onboarding_url:" + ccf_api_onboarding_url)
    logger.debug("ccf_discover_url:" + ccf_discover_url)
    logger.debug("ccf_publish_url:" + ccf_publish_url)
    logger.debug("ccf_security_url:" + ccf_security_url)

    # Onboard the provider
    prov = onboard_provider("MECSandbox_to_CAPIF_Provider", access_token)
    if len(prov) == 0:
        return

    # Print certificates for each function
    logger.debug("API Provider Id:" + prov['apiProvDomId'])
    logger.debug("AMF: " + prov['apiProvFuncs'][0]['regInfo']['apiProvCert'])
    logger.debug("AEF: " + prov['apiProvFuncs'][1]['regInfo']['apiProvCert'])
    logger.debug("APF: " + prov['apiProvFuncs'][2]['regInfo']['apiProvCert'])
    logger.debug("csr: " + str(prov['csr']))

    time.sleep(5) # Sleep for 5 seconds

    # Offboard the API profider
    certs_bundle = (prov['apiProvFuncs'][0]['regInfo']['apiProvCert'], prov['csr']['amf'][1]) # Use AMF certificate and AMF private key
    file_bundle = offboard_provider(prov['apiProvDomId'], certs_bundle)
    if len(file_bundle) == 0:
        for file in file_bundle:
            os.remove(file)

    # Delete the newly created user
    delete_user(user_uuid, admin_token)

    # Logout
    process_logout()

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Using ETSI MEC profile for CAPIF

The purpose is to export the MEC Profile for CAPIF API into our CAPIF application. To achieve it, we need to fulfill the following requirements:
1. Create an instance of a MEC Sandbox using the '4g-5g-macri-v2x' network scenario
2. Set TRY_MEC_URL, TRY_MEC_SESSION_ID, TRY_MEC_PLTF, MEC_APP_INST_ID constant accordingly
3. Build the ServiceAPIDescription as described in ETSI TS 129 222 V18.6.0 (2022-06) Table 8.2.4.2.2-1: Definition of type ServiceAPIDescription. This is the role of the function below

In [ ]:
def build_publish_api_from_mec_services(p_aefId: str) -> dict:
    """
    This function builds the Publish API request body data structure which will be used todo the request for publish API
    :param p_aefId: The AEF ID
    :return The request body data structure on success, an empty dictionary otherwise
    """
    global logger, TRY_MEC_URL, TRY_MEC_SESSION_ID, TRY_MEC_PLTF

    logger.debug('>>> build_publish_api_from_mec_services: p_aefId=' + p_aefId)

    # Sanity checks
    if len(p_aefId) == 0:
        logger.error('build_publish_api_from_mec_services: p_aefId is empty')
        return dict()

    # Build the service-apis data structure
    publish_api_req_body = {
        "apiName": "MEC Profile for CAPIF",
        "aefProfiles": [
            {
            "aefId": p_aefId,
            "versions": [
                {
                "apiVersion": "v1",
                "expiry": "2025-11-30T10:32:02.004Z",
                "resources": [
                    {
                        "resourceName": "MEC Profile of CAPIF",
                        "commType": "REQUEST_RESPONSE",
                        "uri": f"/{TRY_MEC_SESSION_ID}/{TRY_MEC_PLTF}/service-apis/v1/allServiceAPIs",
                        "custOpName": "string",
                        "operations": [
                            "GET"
                        ],
                        "description": "Endpoint to access MEC services"
                    }
                ],
                "custOperations": [
                    {
                        "commType": "REQUEST_RESPONSE",
                        "custOpName": "string",
                        "operations": [
                            "GET"
                        ],
                        "description": "string"
                    }
                ]
                }
            ],
            "protocol": "HTTP_1_1",
            "dataFormat": "JSON",
            "securityMethods": ["OAUTH"],
            "interfaceDescriptions": [
                {
                "ipv4Addr": TRY_MEC_URL,
                "securityMethods": ["OAUTH"]
                }
            ]
            }
        ],
        "description": "MEC Profile of CAPIF",
        "supportedFeatures": "fffff",
        "shareableInfo": {
            "isShareable": True,
            "capifProvDoms": [
                "string"
            ]
        },
        "serviceAPICategory": "string",
        "apiSuppFeats": "fffff",
        "pubApiPath": {
            "ccfIds": [
                "string"
            ]
        },
        "ccfId": "string",
        "apiStatus":{
            "aefIds": [
                p_aefId
            ]
        }
    }

    logger.debug('<<< build_publish_api_from_mec_services: ' + str(publish_api_req_body))
    return publish_api_req_body
    # End of build_publish_api_from_mec_services function

Having built the ServiceAPIDescription data structure, the next step is to implement the CAPIF publish API.

To proceed, we need to enable the TLS mutual authentication using the security material obtained during the onboarding APIs operation ([Onboarding APIs](#onboarding_apis)), i.e. the AEF certificate and the AEF private key ([Generate certificates](#Generate_certificates)).


**Refer to:** ETSI TS 129 222 V18.6.0 (2022-06) Clauses 5.3 CAPIF_Publish_Service_API and 8.2 CAPIF_Publish_Service_API

Before to proceed with the steps above, let's create 2 helper functions to simpily the implemantation of the CAPIF publish API. These helper functions cover the following operations:
- Onboarding operations
- Offboarding operations

#### Onboarding operations

The Onboarding operations include th following steps:
- login
- create a new user
- Get the information to use CAPIF (security materials & URLs)
- onboard the provider


In [ ]:
def onboarding_provider() -> dict:
    """
    To onboard the provider using CAPIF endpoint. It includes:
    - login
    - create a new user
    - Get the information to use CAPIF (security materials & URLs)
    - onboard the provider
    :return: A dictionary containing security material and additional context information on success, or an empty dictionary otherwise
    """
    global logger, ca_root, ccf_api_onboarding_url, ccf_publish_url, ccf_discover_url, ccf_security_url, ccf_onboarding_url

    # Login
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        return dict()

    # Create a new user
    user_name, user_uuid = create_user(admin_token)
    if len(user_uuid) == 0:
        return dict()

    auth = get_auth(user_name, USER_PASSWORD)
    if len(auth) == 0:
        return dict()

    # Set the CAPIF access information
    access_token = auth['access_token']
    ca_root = auth['ca_root']
    ccf_api_onboarding_url = auth['ccf_api_onboarding_url']
    ccf_discover_url = auth['ccf_discover_url']
    ccf_onboarding_url = auth['ccf_onboarding_url']
    ccf_publish_url = auth['ccf_publish_url']
    ccf_security_url = auth['ccf_security_url']

    # Onboard the provider
    prov = onboard_provider("MECSandbox_to_CAPIF_Provider", access_token)
    if len(prov) == 0:
        return dict()

    # Add context data
    prov['refresh_token'] = refresh_token
    prov['admin_token'] = admin_token
    prov['user_uuid'] = user_uuid
    prov['access_token'] = access_token

    return prov
    # End of onboarding_provider function

#### Offboarding operations

The Offboarding operations include th following steps:
- Offboard the API provide
- Delete the user
- Logout


In [ ]:
def offboarding_provider(p_user_uuid: str, p_api_provider_id: str, p_bundle: tuple, p_admin_token: str) -> int:
    """
    To offboard the provider. It includes:
    - Offboard the API provider
    - Delete the user
    - Logout
    :return: 0 on success, or -1 otherwise
    """
    global logger, ccf_api_onboarding_url, access_token

    logger.debug('>>> offboarding_provider: ' + p_user_uuid)

    # Offboard the API profider
    file_bundle = offboard_provider(p_api_provider_id, p_bundle)
    if len(file_bundle) == 0: # Remove cert files if any
        for file in file_bundle:
            os.remove(file)

    # Delete the newly created user
    delete_user(p_user_uuid, p_admin_token)

    # Logout
    process_logout()

    return 0
    # End of offboarding_provider function

#### Publish CAPIF API function

As mentionned above , the prupose of this function is to publish an API, using the TLS mutual authentication. To do so, we need the APF certificate (public keys) and its private key for the signature and the encription processes, and the CA certificate for the verification of the peer certifcate.

**Note**: The http.request function required taht the cerficates and the keys are stored on files, not in memory. So, when our CAPIF applicate terminates, these files shall be removed (freeing resource step). This is the reason the publish_capif_api() function return a tuple containing the files created for the TLS mutual authentication operation.


In [ ]:
def publish_capif_api(p_apfId: str, p_body: dict, p_bundle: tuple) -> list:
    """
    This function is to publish an API on CAPIF server
    :param p_apfId: The APF identifier
    :param p_body: The request body
    :param p_bundle: The bundle of certificates and keys for the TLS mutual authentication operations
    :return: A list containing the files created for the TLS mutual authentication operations on success, or an empty list otherwise
    """
    global logger, ccf_publish_url, ca_root

    logger.debug('>>> publish_capif_api')

    # Sanity checks
    if len(p_bundle) != 2:
        logger.error('publish_capif_api: p_bundle is malformed')
        return []

    try:
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_publish_url.replace('<apfId>', p_apfId)
        logger.debug('publish_capif_api: url=' + url)
        headers = {'Content-Type': 'application/json'}
        logger.debug('publish_capif_api (step1): headers: ' + str(headers))
        logger.debug('publish_capif_api (step2): body: ' + str(p_body))
        bundle = store_certificate_2_files(p_bundle[0], p_bundle[1], ca_root) # Use CA certificate for verif
        if len(bundle) != 3:
            logger.error(f"Error converting in-memory bundle into files")
            return []
        logger.debug('publish_capif_api (step3): bundle: ' + str(bundle))
        response = requests.post(url, headers=headers, data=json.dumps(p_body), cert=(bundle[0], bundle[1]), verify=bundle[2])
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('publish_capif_api (step4): result: ' + str(response.json()))
        if response.status_code != 201:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return []
        res = json.loads(response.text)
        logger.debug('publish_capif_api (step5): res: ' + str(res))
        api_id = res['apiId']
        return bundle
    except requests.exceptions.RequestException as e:
        logger.error(f"publish_capif_api failed: {e}")

    return []
    # End of function publish_capif_api


### Putting everything together



In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the fourth sprint of our CAPIF/MEC application:
    - Onboarding operations
    - Offboarding operations
    """
    global logger, ca_root

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    prov = onboarding_provider()
    if len(prov) == 0:
        return
    user_uuid = prov['user_uuid']

    # Build the publish_api body request
    aefId = prov['apiProvFuncs'][1]['apiProvFuncId']
    apfId = prov['apiProvFuncs'][2]['apiProvFuncId']
    publish_api_req_body = build_publish_api_from_mec_services(aefId)
    if len(publish_api_req_body) == 0:
        offboarding_provider(user_uuid, prov['apiProvDomId'], (prov['apiProvFuncs'][0]['regInfo']['apiProvCert'], prov['csr']['amf'][1]), prov['admin_token']) # Use AMF certificate and AMF private key
        return
    logger.debug("publish_api_req_body: " + str(publish_api_req_body))
    certs_bundle = (prov['apiProvFuncs'][2]['regInfo']['apiProvCert'], prov['csr']['apf'][1]) # Use APF certificate and APF private key

    # Publish the APIs
    #ipdb.set_trace()
    files_bundle = publish_capif_api(apfId, publish_api_req_body, certs_bundle)
    if len(files_bundle) == 0:
        for file in files_bundle:
            os.remove(file)
        offboarding_provider(user_uuid, prov['apiProvDomId'], (prov['apiProvFuncs'][0]['regInfo']['apiProvCert'], prov['csr']['amf'][1]), prov['admin_token']) # Use AMF certificate and AMF private key
        return

    # Terminate the application
    offboarding_provider(
                         user_uuid,
                         prov['apiProvDomId'],
                         (prov['apiProvFuncs'][0]['regInfo']['apiProvCert'], prov['csr']['amf'][1]),  # Use AMF certificate and AMF private key
                         prov['admin_token']
                         )

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Build an helper function to publish the ETSI MEC profile for CAPIF API

To simply the API invoker process, let's create two helpers functions:
- One to publish the ETSI MEC profile for CAPIF API
- One to remove the previously published API

### Helper to publish API

In [ ]:
def publish_api() -> tuple:
    """
    """
    global logger

    logger.debug('>>> publish_api')

    prov = onboarding_provider()
    if len(prov) == 0:
        return ()
    amf_cert_bundle = (prov['apiProvFuncs'][0]['regInfo']['apiProvCert'], prov['csr']['amf'][1]) # Use AMF certificate and AMF private key

    # Build the publish_api body request
    aefId = prov['apiProvFuncs'][1]['apiProvFuncId']
    apfId = prov['apiProvFuncs'][2]['apiProvFuncId']
    apiId = prov['apiProvDomId']
    publish_api_req_body = build_publish_api_from_mec_services(aefId)
    if len(publish_api_req_body) == 0:
        offboarding_provider(prov['user_uuid'], apiId, amf_cert_bundle, prov['admin_token'])
        return ()
    logger.debug("publish_api_req_body: " + str(publish_api_req_body))
    certs_bundle = (prov['apiProvFuncs'][2]['regInfo']['apiProvCert'], prov['csr']['apf'][1]) # Use APF certificate and APF private key

    # Publish the APIs
    files_bundle = publish_capif_api(apfId, publish_api_req_body, certs_bundle)
    if len(files_bundle) == 0:
        for file in files_bundle:
            os.remove(file)
        offboarding_provider(prov['user_uuid'], apiId, amf_cert_bundle, prov['admin_token']) # Use AMF certificate and AMF private key
        return ()

    logger.debug('publish_api: ' + str((apiId, amf_cert_bundle)))
    return (apiId, amf_cert_bundle, prov['user_uuid'], prov['admin_token'])
    # End of function publish_api


### Helper to remove published API

In [ ]:
def remove_publish_api(p_user_uuid: str, p_api_id: str, p_bundle: tuple, p_admin_token: str):
    """
    To remove published API.
    :param p_user_uuid: The user identifier
    :param p_api_id: The API identifier
    :param p_bundle: The bundle of certificates and keys for the TLS mutual authentication operations
    """
    global logger

    logger.debug('>>> remove_publish_api ')

    # Terminate the application
    offboarding_provider(p_user_uuid, p_api_id, p_bundle, p_admin_token)

    return
    # End of function reove_publish_api


### Putting everything together

Let's test these two helpers functions before to go ahead

In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    To test both helpers functions:
    - publish_api
    - remove_publish_api
    """
    global logger

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Publish the MEC profile for CAPIF API
    res = publish_api()
    if len(res) == 0:
        return

    api_id, bundle, user_uuid, admin_token = res

    time.sleep(5) # Sleep for 5 seconds

    # Remove the MEC profile for CAPIF API
    remove_publish_api(user_uuid, api_id, bundle, admin_token)

    logger.debug('Stopped at ' + time.strftime('%Y%m%d-%H%M%S'))
    # End of function process_main

if __name__ == '__main__':
    process_main()


## Using the published API: The CAPIF Invoker side

Now that we are able to publish an API on the CAPIF server, the next step is to use it in order to invoke some MEC Services API. To achieve this goal, we have to implement the CAPI Invoker, following these steps:
- Onboard an API invoker
- Discover the published APIs
- Get a MEC Service API (this step requires that a MEC Sandox platform is already running)
- Offboard an API invoker

### Onboard an API invoker


In [ ]:
def onboard_invoker(p_name: str, p_access_token: str) -> dict:
    """
    To onboard the API invoker.
    :param p_name: The name of the invoker
    :return: A dictionary containing security material for each CAPIF endpoint on success, or an empty dictionary otherwise
    """
    global logger, ccf_api_onboarding_url, ccf_onboarding_url

    logger.debug('>>> onboard_invoker: ' + p_name)
    logger.debug('>>> onboard_invoker: ' + p_access_token)

    try:
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_onboarding_url
        logger.debug('onboard_invoker: url=' + url)
        headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + p_access_token}
        logger.debug('onboard_invoker (step1): headers: ' + str(headers))
        # Request body for onboarding the invoker
        invoker_csr_request, invoker_private_key = generate_csr("API Invoker", "ETSI", "Fr")
        data = {
            "notificationDestination" : "http://host.docker.internal:8086/netapp_callback",
            "supportedFeatures" : "fffffff",
            "apiInvokerInformation" : "dummy",
            "websockNotifConfig" : {
                "requestWebsocketUri" : True,
                "websocketUri" : "websocketUri"
            },
            "onboardingInformation" : {
                "apiInvokerPublicKey" : invoker_csr_request.decode("utf-8"),
            },
            "requestTestNotification" : True
            }
        logger.debug('onboard_invoker (step2): body: ' + str(data))
        response = requests.post(url, headers=headers, data=json.dumps(data), verify="ca_root.pem")
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('onboard_invoker (step3): result: ' + str(response.json()))
        if response.status_code != 201:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return dict()
        res = json.loads(response.text)
        # Add an entry for CSRs and private keys for future usage
        res['csr'] = [invoker_csr_request, invoker_private_key]
        return res
    except requests.exceptions.RequestException as e:
        logger.error(f"onboard_invoker failed: {e}")

    return dict()
    # End of function onboard_invoker

### Offboard an API invoker

In [ ]:
def offboard_invoker(p_invoker_id: str, p_bundle: tuple) -> list:
    """
    To offboard the API invoker.
    :param p_invoker_id: The API invoker identifier
    :param p_bundle: The bundle of certificates and keys for the TLS mutual authentication operations
    :return: 0 on success, -1 otherwise
    """
    global logger, ccf_onboarding_url, ca_root

    logger.debug('>>> offboard_invoker: ' + p_invoker_id)

    try:
        # Delete the newly created user
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_onboarding_url + '/' + p_invoker_id
        logger.debug('offboard_invoker: url=' + url)
        headers = {'Content-Type': 'application/json'}
        logger.debug('offboard_invoker (step1): headers: ' + str(headers))
        bundle = store_certificate_2_files(p_bundle[0], p_bundle[1], ca_root) # Use CA certificate for verif
        if len(bundle) != 3:
            logger.error(f"Error converting in-memory bundle into files")
            return []
        logger.debug('offboard_invoker (step2): bundle: ' + str(bundle))
        response = requests.delete(url, headers=headers, cert=(bundle[0], bundle[1]), verify=bundle[2])
        logger.debug('offboard_invoker (step3): response=' + str(response))
        response.raise_for_status()  # Raise an exception for bad status codes
        if response.status_code != 204:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return []
    except requests.exceptions.RequestException as e:
        logger.error(f"offboard_invoker failed: {e}")
        return []

    return bundle
    # End of function offboard_invoker

### Discover published APIs

In [ ]:
def discover(p_invoker_id: str, p_bundle: tuple, p_access_token: str) -> dict:
    """
    To discover the APIs published by capif core.
    :param p_invoker_id: The API invoker identifier
    :return: A dictionary containing the APIs published by capif core on success, or an empty dictionary otherwise
    """
    global logger, ca_root, ccf_discover_url

    logger.debug('>>> Discover APIs published by capif core')

    try:
        url = 'https://' + CAPIF_HOSTNAME + ':' + str(CAPIF_PORT) + '/' + ccf_discover_url + p_invoker_id
        logger.debug('Discover: url=' + url)
        headers = {'Content-Type': 'application/json', 'Authorization': 'Bearer ' + p_access_token}
        logger.debug('Discover (step1): headers: ' + str(headers))
        bundle = store_certificate_2_files(p_bundle[0], p_bundle[1], ca_root) # Use CA certificate for verif
        if len(bundle) != 3:
            logger.error(f"Error converting in-memory bundle into files")
            return dict()
        logger.debug('Discover (step2): bundle: ' + str(bundle))
        response = requests.get(url, headers=headers, cert=(bundle[0], bundle[1]), verify=bundle[2])
        logger.debug('Discover (step3): response=' + str(response))
        response.raise_for_status()  # Raise an exception for bad status codes
        logger.debug('Discover : result: ' + str(response.json()))
        if response.status_code != 200:
            logger.error(f"Discovery failed: {response.status_code} - {response.text}")
            return dict()
        res = json.loads(response.text)
        return res
    except requests.exceptions.RequestException as e:
        logger.error(f"Discovery failed: {e}")

    return dict()
    # End of function discover

def extract_ipv4_and_uri(p_response_data: json.loads) -> dict:
    # Extract ipv4Addr using a list comprehension
    ipv4_addrs = [
        desc.get("ipv4Addr")
        for profile in p_response_data.get("serviceAPIDescriptions", [])
        for aef in profile.get("aefProfiles", [])
        for desc in aef.get("interfaceDescriptions", [])
    ]

    # Extract uri using a list comprehension
    uris = [
        resource.get("uri")
        for profile in p_response_data.get("serviceAPIDescriptions", [])
        for aef in profile.get("aefProfiles", [])
        for version in aef.get("versions", [])
        for resource in version.get("resources", [])
    ]

    return {"ipv4Addr": ipv4_addrs, "uri": uris}
    # End of function extract_ipv4_and_uri

### Putting everything together

In [ ]:
%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the fiveth sprint of our CAPIF/MEC application:
    - Publish the MEC profile for CAPIF API
    - Create a new user for the invoker
    - Get certificates
    - Onboad the API invoker
    - Do the discovery
    - Offboard the API invoker
    - Delete the
    - Logout the invoker user
    - Remove the MEC profile for CAPIF API
    """
    global logger, ca_root, ccf_api_onboarding_url, ccf_discover_url, ccf_onboarding_url, ccf_publish_url, ccf_security_url

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Publish the MEC profile for CAPIF API
    res = publish_api()
    if len(res) == 0:
        return
    api_id, bundle, prov_user_uuid, prov_admin_token = res

    # Login for the new user for the invoker
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Create a new user for the invoker
    res = create_user(admin_token)
    if len(res) == 0:
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return
    user_name, user_uuid = res

    # Get certificates
    auth = get_auth(user_name, USER_PASSWORD)
    if len(auth) == 0:
        delete_user(user_name, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Sanity checks
    if auth['ca_root'] != ca_root:
        raise Exception('CA root mismatch')
    if  auth['ccf_api_onboarding_url'] != ccf_api_onboarding_url:
        raise Exception('CCF API onboarding URL mismatch')
    if auth['ccf_discover_url'] != ccf_discover_url:
        raise Exception('CCF discover URL mismatch')
    if auth['ccf_onboarding_url'] != ccf_onboarding_url:
        raise Exception('CCF onboarding URL mismatch')
    if auth['ccf_publish_url'] != ccf_publish_url:
        raise Exception('CCF publish URL mismatch')
    if auth['ccf_security_url'] != ccf_security_url:
        raise Exception('CCF security URL mismatch')
    access_token = auth['access_token']

    # Onboad the API invoker
    res = onboard_invoker('API Invoker', access_token)
    if len(res) == 0:
        delete_user(user_uuid, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Do the discovery
    invoker_id = res['apiInvokerId']
    certs_bundle = (res['onboardingInformation']['apiInvokerCertificate'], res['csr'][1])
    mec_api = discover(invoker_id, certs_bundle, access_token)
    if len(mec_api) == 0:
        files_bundle = offboard_invoker(invoker_id, certs_bundle)
        if len(files_bundle) == 0:
            for file in files_bundle:
                os.remove(file)
        delete_user(user_uuid, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Extract the URL to access to the MEC Sandbox platform
    addrs = extract_ipv4_and_uri(mec_api)
    logger.debug('addrs: ' + str(addrs))

    time.sleep(5) # Sleep for 5 seconds

    # Offboard the API invoker
    files_bundle = offboard_invoker(invoker_id, certs_bundle)
    if len(files_bundle) == 0:
        for file in files_bundle:
            os.remove(file)

    # Delete the invoker user
    delete_user(user_uuid, admin_token)

    # Logout the invoker user
    process_logout()

    # Remove the MEC profile for CAPIF API
    remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)

if __name__ == '__main__':
  process_main()

## Discoverig MEC services


### Invoking the MEC profile for CAPIF

AFter discovering the published API, we have the information (see content of addrs data structure in previous execution) to do a request to an existing MEC Sandbox platform to get the list of the MEC services exposed (see TRY_MEC_SESSION_ID).

In [ ]:
def discovering_mec_services(p_url: str) -> dict:
    """
    To discover MEC services API
    :param p_url: The URL to access to the MEC Sandbox platform
    :return: A dictionary containing the MEC services on success, or an empty dictionary otherwise
    """
    global logger

    logger.debug('>>> discovering_mec_services: ' + p_url)

    try:
        headers = {'Content-Type': 'application/json', 'accept': 'application/json',}
        logger.debug('discovering_mec_services (step1): headers: ' + str(headers))
        response = requests.get(p_url, headers=headers)
        logger.debug('discovering_mec_services (step2): result: ' + str(response.json()))
        response.raise_for_status()  # Raise an exception for bad status codes
        if response.status_code != 200:
            logger.error(f"Error creating user: {response.status_code} - {response.text}")
            return dict()
        res = json.loads(response.text)
        return res
    except requests.exceptions.RequestException as e:
        logger.error(f"discovering_mec_services failed: {e}")

    return dict()
    # End of function discovering_mec_services

### Putting everything together

Here is the last and complete version of our CAPIF application achieving the main objective of this tutorial: retrieve the MEC services exposed by an existing MEC Sandbox platform.

In [ ]:
#%%script echo skipping
# Uncomment the line above to skip execution of this cell
def process_main():
    """
    This is the fiveth sprint of our CAPIF/MEC application:
    - Publish the MEC profile for CAPIF API
    - Create a new user for the invoker
    - Get certificates
    - Onboad the API invoker
    - Do the discovery
    - Offboard the API invoker
    - Delete the
    - Logout the invoker user
    - Remove the MEC profile for CAPIF API
    """
    global logger, ca_root, ccf_api_onboarding_url, ccf_discover_url, ccf_onboarding_url, ccf_publish_url, ccf_security_url

    logger.debug('Starting at ' + time.strftime('%Y%m%d-%H%M%S'))
    logger.debug('\t pwd= ' + os.getcwd())

    # Publish the MEC profile for CAPIF API
    res = publish_api()
    if len(res) == 0:
        return
    api_id, bundle, prov_user_uuid, prov_admin_token = res

    # Login for the new user for the invoker
    refresh_token, admin_token = process_login()
    if refresh_token is None:
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Create a new user for the invoker
    res = create_user(admin_token)
    if len(res) == 0:
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return
    user_name, user_uuid = res

    # Get certificates
    auth = get_auth(user_name, USER_PASSWORD)
    if len(auth) == 0:
        delete_user(user_name, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Sanity checks
    if auth['ca_root'] != ca_root:
        raise Exception('CA root mismatch')
    if  auth['ccf_api_onboarding_url'] != ccf_api_onboarding_url:
        raise Exception('CCF API onboarding URL mismatch')
    if auth['ccf_discover_url'] != ccf_discover_url:
        raise Exception('CCF discover URL mismatch')
    if auth['ccf_onboarding_url'] != ccf_onboarding_url:
        raise Exception('CCF onboarding URL mismatch')
    if auth['ccf_publish_url'] != ccf_publish_url:
        raise Exception('CCF publish URL mismatch')
    if auth['ccf_security_url'] != ccf_security_url:
        raise Exception('CCF security URL mismatch')
    access_token = auth['access_token']

    # Onboad the API invoker
    res = onboard_invoker('API Invoker', access_token)
    if len(res) == 0:
        delete_user(user_uuid, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Do the discovery
    invoker_id = res['apiInvokerId']
    certs_bundle = (res['onboardingInformation']['apiInvokerCertificate'], res['csr'][1])
    mec_api = discover(invoker_id, certs_bundle, access_token)
    if len(mec_api) == 0:
        files_bundle = offboard_invoker(invoker_id, certs_bundle)
        if len(files_bundle) == 0:
            for file in files_bundle:
                os.remove(file)
        delete_user(user_uuid, admin_token)
        remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)
        return

    # Extract the URL to access to the MEC Sandbox platform
    addrs = extract_ipv4_and_uri(mec_api)
    logger.debug('addrs: ' + str(addrs))

    # Discovering MEC services
    url = ''
    if 'ports' in addrs:
        url = 'https://' + addrs['ipv4Addr'][0] + ':' + addrs['ports'][0]
    else:
        url = 'https://' + addrs['ipv4Addr'][0]
    url += addrs['uri'][0]
    mec_services = discovering_mec_services(url)
    if len(mec_services) != 0:
        logger.debug('===> The list of the MEC services exposed by ' + addrs['ipv4Addr'][0] + ' is: ' + str(mec_services))

    # Offboard the API invoker
    files_bundle = offboard_invoker(invoker_id, certs_bundle)
    if len(files_bundle) == 0:
        for file in files_bundle:
            os.remove(file)

    # Delete the invoker user
    delete_user(user_uuid, admin_token)

    # Logout the invoker user
    process_logout()

    # Remove the MEC profile for CAPIF API
    remove_publish_api(prov_user_uuid, api_id, bundle, prov_admin_token)

if __name__ == '__main__':
  process_main()

### What to do next

here is a list of several additional tasks yo can do to refine our CAPIF application:
1. Simply the code of the process_main() function above
2. Find the endpoint of the MEC Location API service (MEC 013) and send a request to get the list of zones available
3. Create your own CAPIF application

## Conlusion: what do we learn?

The main objective of this tutorial is to demonstrate how to use the MEC profile for CAPIF (as described in MEC 011 v3.x.x Clause 9) in a CAPIF application. Along this tutrial, we learned how to develop a basic CAPIF application, including both API provider and API invoker. We learned also how to use a published API to send REQUEST to a MEC Sandbox platform instance.


**That's all Folks**

